# SALT on Chameleon Cloud (PACT 2026 Reproducibility Challenge)

This notebook provisions the required Chameleon hardware, configures it, launches the released SALT Docker artifact, downloads the results, and releases all resources. Run the cells in order. The normal `smoke` and `reproduce` paths use the checked-in reference PMC data and require no infrastructure beyond the node created here.

**Important:** download every result set you want to keep before running the final teardown cell. The bare-metal node is ephemeral.

## Fixed hardware and image

| Setting | Value |
| --- | --- |
| Site | `CHI@UC` |
| Node type | `compute_skylake` |
| Expected processor | Intel Xeon Gold 6126 (Skylake-SP) |
| Nodes / GPU | 1 bare-metal node / no GPU |
| Image | `CC-Ubuntu24.04` |
| Lease | 16 hours; explicitly released during teardown |
| Artifact | `salt-pact26-v1.1.tar.gz` |

The full workflow can take about 8--9 hours after a clean Docker build, which can itself take 1--2 hours. The lease includes safety margin; teardown releases it early.

## 1. Configure the Chameleon project

The project chooser below is the only interactive infrastructure selection. Hardware, image, network, and duration are fixed by the artifact.

In [ ]:
from datetime import timedelta
from pathlib import Path
import getpass
import hashlib
import os
import re
import shlex
import time

from chi import context, lease, server

SITE = "CHI@UC"
NODE_TYPE = "compute_skylake"
IMAGE_NAME = "CC-Ubuntu24.04"
LEASE_HOURS = 16
ARTIFACT_ARCHIVE = Path("salt-pact26-v1.1.tar.gz").resolve()
ARTIFACT_SHA256 = "53d8df9da4fc714b86f9fb884f4fcd8b1ed4c0cbec65a401409fd951f5e5b559"
SETUP_SCRIPT = Path("chameleon/setup-node.sh").resolve()
REMOTE_HOME = "/home/cc"
REMOTE_SOURCE = f"{REMOTE_HOME}/salt-artifact"
REMOTE_RESULTS = f"{REMOTE_HOME}/salt-results"
DOCKER_IMAGE = "salt-artifact:v1.1"

raw_owner = os.environ.get("USER") or getpass.getuser()
owner = re.sub(r"[^a-z0-9-]+", "-", raw_owner.lower()).strip("-")
LEASE_NAME = f"{owner}-salt-pact26"
SERVER_NAME = LEASE_NAME

assert ARTIFACT_ARCHIVE.is_file(), f"Missing {ARTIFACT_ARCHIVE}"
assert SETUP_SCRIPT.is_file(), f"Missing {SETUP_SCRIPT}"
context.use_site(SITE)
context.choose_project()
print(f"Lease/server name: {LEASE_NAME}")
print(f"Artifact: {ARTIFACT_ARCHIVE}")

## 2. Reserve the node and floating IP

The submission is idempotent: rerunning this cell reconnects to an active lease with the same name rather than reserving another node.

In [ ]:
l = lease.Lease(name=LEASE_NAME, duration=timedelta(hours=LEASE_HOURS))
l.add_node_reservation(amount=1, node_type=NODE_TYPE)
l.add_fip_reservation(amount=1)
l.submit(wait_for_active=True, wait_timeout=900, show="text", idempotent=True)
assert l.node_reservations, "The active lease has no node reservation"
print(f"Lease ID: {l.id}; status: {l.status}; ends: {l.end_date}")

## 3. Launch Ubuntu and wait for SSH

In [ ]:
s = server.Server(
    name=SERVER_NAME,
    reservation_id=l.node_reservations[0]["id"],
    image_name=IMAGE_NAME,
)
s.submit(
    wait_for_active=True,
    wait_timeout=1200,
    show="text",
    idempotent=True,
    retry_on_error=True,
)
floating_ip = l.get_reserved_floating_ips()[0]
s.refresh()
if floating_ip not in s.get_all_floating_ips():
    s.associate_floating_ip(floating_ip)
    s.refresh()
s.check_connectivity(host=floating_ip, timeout=900, show="text")
print(f"Server ID: {s.id}; floating IP: {floating_ip}")

### Recovery after a notebook-kernel restart

If the kernel restarts while the lease is still active, rerun the configuration cell and then this cell. Otherwise continue below.

In [ ]:
l = lease.get_lease(LEASE_NAME)
assert l is not None, f"No active lease named {LEASE_NAME}"
s = server.get_server(SERVER_NAME)
floating_ip = s.get_floating_ip()
s.check_connectivity(host=floating_ip, timeout=900, show="text")
print(f"Recovered lease {l.id} and server {s.id} at {floating_ip}")

## 4. Define transfer helpers and record the allocated hardware

In [ ]:
def run_remote(command: str):
    return s.execute(command)

def download_tree(remote_directory: str, local_archive: str) -> Path:
    local_path = Path(local_archive).resolve()
    remote_archive = f"{REMOTE_HOME}/{local_path.name}"
    command = (
        f"test -d {shlex.quote(remote_directory)} && "
        f"tar -C {shlex.quote(remote_directory)} -czf {shlex.quote(remote_archive)} ."
    )
    run_remote(command)
    with s.ssh_connection() as connection:
        connection.get(remote_archive, str(local_path))
    print(f"Downloaded {local_path} ({local_path.stat().st_size:,} bytes)")
    return local_path

In [ ]:
hardware = run_remote(
    "set -e; uname -a; echo; lscpu; echo; free -h; echo; lsblk; echo; df -h /"
)
Path("chameleon-node-hardware.txt").write_text(hardware.stdout)
print("Saved chameleon-node-hardware.txt in the Trovi workspace.")

## 5. Configure the host and stage the immutable release

The setup script installs standard LLVM/MLIR/ChampSim build prerequisites, `perf`, and Docker Engine; enables Docker; adds `cc` to the Docker group; and permits `perf_event_open` on this dedicated leased node. The notebook uses `sudo docker` so it does not depend on group refresh timing.

In [ ]:
s.upload(str(SETUP_SCRIPT), remote_path=f"{REMOTE_HOME}/setup-node.sh")
run_remote(f"chmod 0755 {REMOTE_HOME}/setup-node.sh && sudo {REMOTE_HOME}/setup-node.sh cc")

In [ ]:
local_sha256 = hashlib.sha256(ARTIFACT_ARCHIVE.read_bytes()).hexdigest()
assert local_sha256 == ARTIFACT_SHA256, f"Unexpected release archive SHA-256: {local_sha256}"
remote_archive = f"{REMOTE_HOME}/{ARTIFACT_ARCHIVE.name}"
s.upload(str(ARTIFACT_ARCHIVE), remote_path=remote_archive)
remote_sha256 = run_remote(f"sha256sum {shlex.quote(remote_archive)}").stdout.split()[0]
assert remote_sha256 == ARTIFACT_SHA256, (ARTIFACT_SHA256, remote_sha256)
run_remote(
    f"set -e; "
    f"if [ ! -f {REMOTE_SOURCE}/.salt-v1.1 ]; then "
    f"test ! -e {REMOTE_SOURCE}; mkdir -p {REMOTE_SOURCE}; "
    f"tar -xzf {shlex.quote(remote_archive)} -C {REMOTE_SOURCE} --strip-components=1; "
    f"touch {REMOTE_SOURCE}/.salt-v1.1; fi"
)
print(f"Verified release SHA-256: {local_sha256}")

## 6. Build and validate the Docker artifact

The first build downloads LLVM/MLIR, Rust, and Python dependencies and may take 1--2 hours. Its toolchain versions are pinned by the release.

In [ ]:
run_remote(
    f"cd {REMOTE_SOURCE} && "
    f"sudo docker build --progress=plain --tag {DOCKER_IMAGE} ."
)

In [ ]:
run_remote(f"sudo docker run --rm --init {DOCKER_IMAGE} test")

## 7A. Recommended first run: end-to-end smoke reproduction

This reduced run exercises all three experiment workflows and should take about 30 seconds once the image exists. It uses the packaged reference PMCs and writes results on the node before downloading them here. Each result path must be new to prevent accidental mixing of runs.

In [ ]:
smoke_results = f"{REMOTE_RESULTS}/smoke"
run_remote(
    f"set -e; test ! -e {smoke_results}; mkdir -p {smoke_results}; "
    f"sudo docker run --rm --init "
    f"--volume {smoke_results}:/artifact/results "
    f"{DOCKER_IMAGE} smoke"
)

In [ ]:
download_tree(smoke_results, "salt-chameleon-smoke-results.tar.gz")

## 7B. Full paper reproduction

Run the next two cells for the complete evaluation. Budget approximately 8--9 hours after the image build. This is independent of the smoke result directory. Keep this browser/kernel session available while the remote command streams output.

In [ ]:
full_results = f"{REMOTE_RESULTS}/full"
run_remote(
    f"set -e; test ! -e {full_results}; mkdir -p {full_results}; "
    f"sudo docker run --rm --init "
    f"--volume {full_results}:/artifact/results "
    f"{DOCKER_IMAGE} reproduce"
)

In [ ]:
download_tree(full_results, "salt-chameleon-full-results.tar.gz")

## 8. Optional: collect fresh PMCs on the Xeon Gold 6126

This is an additional machine-specific experiment, not the portable Figure 4 reproduction above. The Xeon Gold 6126 is Skylake-SP, whereas the paper's checked-in measurements came from a Core i7-7700 (Kaby Lake). The generic L1D-read-miss event is available on both, but absolute counts can differ because of CPU, compiler, frequency, memory, and system effects. Keep these results separately named.

The first cell disables SMT on this dedicated bare-metal lease and selects an online nonzero logical CPU. The second grants only the container permissions needed for `perf_event_open`, collects three repeats, and compares those measurements to SALT using the same 32 KiB / 64-byte L1D geometry.

In [ ]:
run_remote(
    "set -e; "
    "if [ -e /sys/devices/system/cpu/smt/control ]; then "
    "echo off | sudo tee /sys/devices/system/cpu/smt/control >/dev/null; fi; "
    "lscpu -e=CPU,CORE,SOCKET,ONLINE"
)
cpu_query = (
    "lscpu -p=CPU,ONLINE | "
    "awk -F, '$1 !~ /^#/ && $2 == \"Y\" && $1 != \"0\" {print $1; exit}'"
)
measurement_cpu = int(run_remote(cpu_query).stdout.strip())
print(f"Fresh measurements will be pinned to logical CPU {measurement_cpu}.")

In [ ]:
pmc_root = f"{REMOTE_RESULTS}/pmc-skylake"
pmc_command = (
    f"python3 salt_vs_hw_misses_package/benchmarks/collect_pmc.py "
    f"--cpu {measurement_cpu} --repeats 3 "
    f"--output /artifact/results/pmc-skylake.csv && "
    f"RESULTS_DIR=/artifact/results/comparison "
    f"./scripts/run-salt-vs-hardware-evaluation.sh "
    f"--pmc /artifact/results/pmc-skylake.csv"
)
run_remote(
    f"set -e; test ! -e {pmc_root}; mkdir -p {pmc_root}; "
    f"sudo docker run --rm --init "
    f"--cap-add PERFMON --security-opt seccomp=unconfined "
    f"--volume {pmc_root}:/artifact/results "
    f"{DOCKER_IMAGE} shell -lc {shlex.quote(pmc_command)}"
)
download_tree(pmc_root, "salt-chameleon-skylake-pmc-results.tar.gz")

## 9. Required teardown

Confirm that the wanted result archives exist in the Jupyter/Trovi file browser, then run this cell. It deletes the server, waits until Nova no longer lists it, and deletes the lease. Deleting the lease releases both the `compute_skylake` node and its reserved floating IP instead of waiting for lease expiry. The cell can also be run after kernel recovery.

In [ ]:
matching_servers = [item for item in server.list_servers() if item.name == SERVER_NAME]
for item in matching_servers:
    print(f"Deleting server {item.name} ({item.id})...")
    item.delete(idempotent=True, delete_ips=False)

deadline = time.monotonic() + 300
while time.monotonic() < deadline:
    if not any(item.name == SERVER_NAME for item in server.list_servers()):
        break
    time.sleep(5)
else:
    raise TimeoutError("Server deletion was not confirmed; do not delete the lease yet.")

matching_leases = [
    item for item in lease.list_leases()
    if item.name == LEASE_NAME
    and str(item.status).upper() not in {"TERMINATED", "DELETED"}
]
for item in matching_leases:
    print(f"Deleting lease {item.name} ({item.id})...")
    item.delete()
print("Teardown complete: server, node reservation, and floating-IP reservation released.")